# Supernova neutrino classification

Binary CNN classification of simulated liquid-argon detector images under electronic noise and a Gaussian-blob background model. The target is the presence of a neutrino event: **1 = signal, 0 = background**.

The workflow covers noise simulation, classifier training, noise-level scans and misclassified-event inspection. See [README](README.md) for data access and installation.

In [ ]:
from pathlib import Path
import os

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

DATA_DIR = Path(os.environ.get('SUPERNOVA_DATA_DIR', 'data')).expanduser()
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
SEED = 42
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True})

## Data and event separation

The supplied dataset contains 10,000 images of shape `(100, 100)` and 64 metadata values per event. Metadata column 1 is neutrino energy in MeV; column 3 is the final-state particle count. Pixel amplitudes and the noise parameters below use the image's numerical scale, without a detector-specific calibration.

Training uses an initial block of events and testing a disjoint final block. For paired blob views, both views of an event stay in the same split. This is an exploratory noise-robustness study; the test scans are not a separate, blind final benchmark.

In [ ]:
images = np.load(DATA_DIR / 'larImages.npy', mmap_mode='r', allow_pickle=False)
meta = np.load(DATA_DIR / 'meta.npy', mmap_mode='r', allow_pickle=False)
if images.shape != (10000, 100, 100) or meta.shape != (10000, 64):
    raise ValueError('Expected larImages.npy (10000, 100, 100) and meta.npy (10000, 64).')
if images.dtype.kind != 'f' or meta.dtype.kind not in 'fi':
    raise ValueError('The image and metadata arrays must be numeric.')

print(f'Images: {images.shape}; metadata: {meta.shape}')
active = np.concatenate([image[image != 0] for image in images])
active_stds = [np.std(image[image != 0]) for image in images if np.any(image != 0)]
print(f'Mean within-image standard deviation of active pixels: {np.mean(active_stds):.2f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(active, bins=80)
ax.set(xlabel='Nonzero pixel amplitude', ylabel='Pixel count', yscale='log')
plt.show()
del active

## Classifier and metrics

Each CNN has two 32-filter `3 × 3` convolutions with ReLU and `2 × 2` max pooling, followed by a 32-unit dense layer, 0.5 dropout and a sigmoid output. Training uses Adam, binary cross-entropy and batches of 300, with no image rescaling. The decision threshold is 0.5.

Accuracy is the correctly classified fraction of all images. Signal recall and background recall are reported separately to distinguish missed events from false positives. Each scan point uses a fresh noise realisation.

In [ ]:
def build_classifier() -> keras.Model:
    '''Build the binary image classifier.'''
    model = keras.Sequential([
        keras.Input(shape=(100, 100, 1)),
        keras.layers.Conv2D(32, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(32, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


def train_classifier(x: np.ndarray, y: np.ndarray, epochs: int) -> keras.Model:
    '''Train a CNN for a fixed number of epochs.
    Args
        x (ndarray): Images with shape (n_images, 100, 100).
        y (ndarray): Binary labels with shape (n_images,).
        epochs (int): Number of training epochs.
    Returns
        Model: Classifier with the final epoch's weights.
    '''
    model = build_classifier()
    dataset = tf.data.Dataset.from_tensor_slices((x[..., None], y))
    dataset = dataset.shuffle(len(y), seed=SEED).batch(300)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    dataset = dataset.with_options(options)
    model.fit(dataset, epochs=epochs, verbose=2, shuffle=False)
    return model


def predict_labels(model: keras.Model, x: np.ndarray) -> np.ndarray:
    '''Predict binary labels in batches of 64.
    Args
        model (Model): Trained classifier.
        x (ndarray): Images with shape (n_images, 100, 100).
    Returns
        ndarray: Predicted labels with shape (n_images,).
    '''
    scores = np.concatenate([
        model(x[start:start + 64, ..., None], training=False).numpy().ravel()
        for start in range(0, len(x), 64)
    ])
    return (scores > 0.5).astype(np.int8)


def evaluate(model: keras.Model, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    '''Calculate accuracy and class-specific recalls.
    Args
        model (Model): Trained classifier.
        x (ndarray): Images with shape (n_images, 100, 100).
        y (ndarray): Binary labels containing both classes.
    Returns
        ndarray: Accuracy, signal recall and background recall, each in [0, 1].
    '''
    predicted = predict_labels(model, x)
    correct = predicted == y
    return np.array([correct.mean(), correct[y == 1].mean(), correct[y == 0].mean()])

## Electronic noise

Electronics noise is independent, zero-mean Gaussian noise with standard deviation `sigma`. C0 learns clean signal and blank backgrounds; C1 learns clean signal and noisy backgrounds; C2 sees noise on both classes. C1 and C2 use `sigma = 5` during training, comparable to the spread of active pixel values.

Training contains 7,000 signal images, 9,000 noisy backgrounds and 1,000 blank backgrounds. Testing contains the last 3,000 signal images and 3,000 independently generated backgrounds. C0 and C1 train for 5 epochs; C2 trains for 100.

In [ ]:
def electronic_dataset(signal: np.ndarray, sigma: float, noisy_signal: bool,
                       n_background: int, n_blank: int = 0) -> tuple[np.ndarray, np.ndarray]:
    '''Combine event images with Gaussian backgrounds.
    Args
        signal (ndarray): Signal images with shape (n_signal, 100, 100).
        sigma (float): Gaussian standard deviation in image-amplitude units.
        noisy_signal (bool): Whether to add noise to the signal images.
        n_background (int): Number of Gaussian background images.
        n_blank (int): Number of additional zero-valued background images.
    Returns
        tuple: Float32 images and binary labels, with signal images first.
    '''
    n_signal = len(signal)
    x = np.zeros((n_signal + n_background + n_blank, 100, 100), dtype=np.float32)
    x[:n_signal] = signal
    if noisy_signal:
        x[:n_signal] += rng.normal(0, sigma, signal.shape).astype(np.float32)
    x[n_signal:n_signal + n_background] = rng.normal(
        0, sigma, (n_background, 100, 100)
    ).astype(np.float32)
    y = np.zeros(len(x), dtype=np.int8)
    y[:n_signal] = 1
    return x, y


electronic_models = {}
for name, sigma, noisy_signal, epochs in [
    ('C0', 0, False, 5), ('C1', 5, False, 5), ('C2', 5, True, 100)
]:
    x_train, y_train = electronic_dataset(images[:7000], sigma, noisy_signal, 9000, 1000)
    print(name)
    electronic_models[name] = train_classifier(x_train, y_train, epochs)
    del x_train, y_train

In [ ]:
noise_levels = np.linspace(0, 20, 10)
clean_results = {name: [] for name in ['C0', 'C1']}
for sigma in noise_levels:
    x_test, y_test = electronic_dataset(images[-3000:], sigma, False, 3000)
    for name in clean_results:
        clean_results[name].append(evaluate(electronic_models[name], x_test, y_test))
    del x_test, y_test

noisy_levels = np.linspace(0, 10, 30)
noisy_results = {name: [] for name in ['C1', 'C2']}
for sigma in noisy_levels:
    x_test, y_test = electronic_dataset(images[-3000:], sigma, True, 3000)
    for name in noisy_results:
        noisy_results[name].append(evaluate(electronic_models[name], x_test, y_test))
    del x_test, y_test

clean_results = {name: np.asarray(values) for name, values in clean_results.items()}
noisy_results = {name: np.asarray(values) for name, values in noisy_results.items()}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for name, values in clean_results.items():
    axes[0].plot(noise_levels, values[:, 0], label=name)
for name, values in noisy_results.items():
    axes[1].plot(noisy_levels, values[:, 0], label=name)
for column, label in [(1, 'Signal'), (2, 'Background')]:
    axes[2].plot(noisy_levels, noisy_results['C2'][:, column], label=label)
for ax, title in zip(axes, ['Clean signal', 'Noisy signal', 'C2 class recall']):
    ax.set(title=title, xlabel='Gaussian noise standard deviation', ylim=(0, 1.03))
    ax.legend()
axes[0].set_ylabel('Accuracy')
axes[1].set_ylabel('Accuracy')
axes[2].set_ylabel('Recall')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'electronic-noise.png', dpi=180)
np.savez(OUTPUT_DIR / 'electronic-scan.npz', clean_levels=noise_levels,
         noisy_levels=noisy_levels, clean_C0=clean_results['C0'],
         clean_C1=clean_results['C1'], noisy_C1=noisy_results['C1'], noisy_C2=noisy_results['C2'])
plt.show()

Training on noisy backgrounds alone does not teach C1 how signal looks under noise. C2 is more robust when both classes contain noise. At larger noise amplitudes, the main loss is signal recall: more events are missed, while most backgrounds remain correctly classified.

## Gaussian-blob backgrounds

A toy radiological background consists of two views of a Gaussian blob sharing the same randomly sampled 3D centre. Both are added to the same underlying event image. In the transverse view, `width` controls both spatial standard deviations. In the second view, the vertical standard deviation is `height`, which also controls the amplitude of both views:

\[
B_{xy}=h\exp\left[-\frac{(x-x_0)^2+(y-y_0)^2}{2w^2}\right],\qquad
B_{xz}=h\exp\left[-\frac{(x-x_0)^2}{2w^2}-\frac{(z-z_0)^2}{2h^2}\right].
\]

`height` therefore couples amplitude and spatial extent. These are shape parameters for a toy model, not calibrated decay energies or predictions for individual isotopes.

In [ ]:
grid_x, grid_y = np.meshgrid(np.arange(100), np.arange(100))


def blob_views(width: float, height: float) -> tuple[np.ndarray, np.ndarray]:
    '''Generate two Gaussian views with a shared three-dimensional centre.
    Args
        width (float): Positive transverse standard deviation in pixels.
        height (float): Positive amplitude and vertical-extent parameter.
    Returns
        tuple: Two float32 background arrays, each with shape (100, 100).
    '''
    if width <= 0 or height <= 0:
        raise ValueError('Blob width and height must be positive.')
    centre_x, centre_y, centre_z = rng.uniform(0, 100, size=3)
    horizontal = (grid_x - centre_x)**2 / (2 * width**2)
    xy = height * np.exp(-horizontal - (grid_y - centre_y)**2 / (2 * width**2))
    xz = height * np.exp(-horizontal - (grid_y - centre_z)**2 / (2 * height**2))
    return xy.astype(np.float32), xz.astype(np.float32)


def blob_dataset(signal: np.ndarray, n_background: int, n_blank: int,
                 width: float | None, height: float | None,
                 noisy_signal: bool) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Combine signal events with paired Gaussian-blob backgrounds.
    Args
        signal (ndarray): Signal images with shape (n_signal, 100, 100).
        n_background (int): Number of background centres; each produces two views.
        n_blank (int): Number of additional zero-valued background images.
        width (float or None): Blob width; None samples uniformly from 0.01 to 40.
        height (float or None): Blob height; None samples uniformly from 0.01 to 40.
        noisy_signal (bool): Whether each signal event produces two noisy views.
    Returns
        tuple: Float32 images, binary labels and source-event indices (-1 for background).
    '''
    n_signal = len(signal)
    signal_count = 2 * n_signal if noisy_signal else n_signal
    x = np.zeros((signal_count + 2 * n_background + n_blank, 100, 100), dtype=np.float32)
    event_index = np.full(len(x), -1, dtype=int)
    if noisy_signal:
        for i, event in enumerate(signal):
            w = rng.uniform(0.01, 40) if width is None else width
            h = rng.uniform(0.01, 40) if height is None else height
            xy, xz = blob_views(w, h)
            x[i], x[n_signal + i] = event + xy, event + xz
        event_index[:signal_count] = np.tile(np.arange(n_signal), 2)
    else:
        x[:n_signal] = signal
        event_index[:n_signal] = np.arange(n_signal)
    for i in range(n_background):
        w = rng.uniform(0.01, 40) if width is None else width
        h = rng.uniform(0.01, 40) if height is None else height
        xy, xz = blob_views(w, h)
        x[signal_count + i] = xy
        x[signal_count + n_background + i] = xz
    y = np.zeros(len(x), dtype=np.int8)
    y[:signal_count] = 1
    return x, y, event_index

Cr0 trains for 6 epochs on 6,000 clean signals, 6,000 blob views (`width=2`, `height=5`) and 1,000 blank images. Its clean-signal scan uses 3,000 signal images and 4,000 background views, so the majority-class accuracy is 4/7.

Cr1, Cr2 and Cr3 train for 6 epochs on paired noisy views of the first 5,000 events, 8,000 background views and 3,000 blanks. Their `(width, height)` settings are `(5, 20)`, `(10, 5)` and `(15, 15)`. Noisy-signal scans use paired views of the last 2,000 events and 4,000 backgrounds, giving balanced classes. Paired views are correlated observations, not independent events.

In [ ]:
x_train, y_train, _ = blob_dataset(images[:6000], 3000, 1000, 2, 5, False)
cr0 = train_classifier(x_train, y_train, epochs=6)
del x_train, y_train

blob_models = {}
for name, width, height in [('Cr1', 5, 20), ('Cr2', 10, 5), ('Cr3', 15, 15)]:
    x_train, y_train, _ = blob_dataset(images[:5000], 4000, 3000, width, height, True)
    print(name)
    blob_models[name] = train_classifier(x_train, y_train, epochs=6)
    del x_train, y_train

In [ ]:
widths = np.linspace(1, 40, 5)
heights = np.linspace(1, 40, 5)
blob_results = {name: np.zeros((len(widths), len(heights)))
                for name in ['Cr0 clean', 'Cr0 noisy', *blob_models]}
for i, width in enumerate(widths):
    for j, height in enumerate(heights):
        x_test, y_test, _ = blob_dataset(images[-3000:], 2000, 0, width, height, False)
        blob_results['Cr0 clean'][i, j] = evaluate(cr0, x_test, y_test)[0]
        del x_test, y_test
        x_test, y_test, _ = blob_dataset(images[-2000:], 2000, 0, width, height, True)
        blob_results['Cr0 noisy'][i, j] = evaluate(cr0, x_test, y_test)[0]
        for name, model in blob_models.items():
            blob_results[name][i, j] = evaluate(model, x_test, y_test)[0]
        del x_test, y_test

fig, axes = plt.subplots(1, 5, figsize=(17, 3), sharey=True)
for ax, (name, values) in zip(axes, blob_results.items()):
    mesh = ax.pcolormesh(widths, heights, values.T, cmap='RdPu', vmin=0.5, vmax=1)
    ax.set(title=name, xlabel='Blob width (pixels)')
    ax.grid(False)
axes[0].set_ylabel('Blob height parameter')
fig.colorbar(mesh, ax=axes, label='Accuracy', shrink=0.9)
fig.savefig(OUTPUT_DIR / 'blob-noise.png', dpi=180, bbox_inches='tight')
np.savez(OUTPUT_DIR / 'blob-scan.npz', widths=widths, heights=heights, **blob_results)
plt.show()

The sensitivity depends on both blob parameters and the training noise configuration. Narrow, tall blobs and broad, high-amplitude blobs can reduce accuracy. This is consistent with confusion between track-like backgrounds and signal, or with signal being obscured by the overlay.

## Misclassified events

CrR uses the same training composition as Cr1–Cr3, with width and height sampled independently from `Uniform(0.01, 40)` for each signal or background centre. After 6 epochs, inspect errors on one independently generated set of 8,000 noisy test views. Metadata is used only for interpreting signal errors, not as a classifier input.

In [ ]:
x_train, y_train, _ = blob_dataset(images[:5000], 4000, 3000, None, None, True)
cr_random = train_classifier(x_train, y_train, epochs=6)
del x_train, y_train

x_test, y_test, event_indices = blob_dataset(images[-2000:], 2000, 0, None, None, True)
predicted = predict_labels(cr_random, x_test)
correct = predicted == y_test
print(f'Accuracy: {correct.mean():.4f}')
print(f'Signal recall: {correct[y_test == 1].mean():.4f}')
print(f'Background recall: {correct[y_test == 0].mean():.4f}')

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row, label in enumerate([1, 0]):
    errors = np.flatnonzero((predicted != y_test) & (y_test == label))
    for ax in axes[row]:
        ax.axis('off')
    for ax, error in zip(axes[row], errors[:4]):
        ax.imshow(x_test[error], cmap='gray')
        if label == 1:
            energy = meta[-2000:][event_indices[error], 1]
            ax.set_title(f'Missed signal: {energy:.1f} MeV')
        else:
            ax.set_title('False positive')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'misclassified-events.png', dpi=180)
plt.show()
del x_test, y_test

## Interpretation

Training noise must resemble the conditions used for evaluation. Class-specific recall exposes failure modes that overall accuracy alone hides. The Gaussian blobs provide a controlled shape study, but do not establish detector efficiency or isotope-specific rejection. Exact curves depend on stochastic training and noise realisations.

The problem setup and image/metadata schema come from the Supernova miniproject materials for *Practical Machine Learning*; the accompanying FAQ is by Cheryl Patrick and Ryan Nichol (December 2020).